# B1 — Pretrain CTC Head on LibriSpeech (Frozen WavLM)

**Goal:** Train only the CTC head (768 → vocab linear layer) on LibriSpeech-clean-100
so it learns general English acoustic-to-character mapping from WavLM representations.

**Why:** The CTC head is randomly initialized. Going directly to TORGO (~2000 utterances)
means the head only ever sees 15 speakers and limited vocabulary. Pre-training on
LibriSpeech gives it exposure to hundreds of speakers and broad English vocabulary.

**Pipeline:**
- B1 (this notebook): Freeze WavLM, train CTC head on LibriSpeech-clean-100
- B2 (next notebook): Unfreeze WavLM, fine-tune everything on TORGO

**Compute:** Only ~22K trainable params (one linear layer), runs fast on T4.

In [1]:
import os

# Force single GPU before any torch import
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

os.environ["HF_HOME"]           = "/kaggle/temp/hf"
os.environ["HF_DATASETS_CACHE"] = "/kaggle/temp/hf/datasets"
os.environ["HF_HUB_CACHE"]      = "/kaggle/temp/hf/hub"
os.environ["TRANSFORMERS_CACHE"] = "/kaggle/temp/hf/transformers"
os.environ["XDG_CACHE_HOME"]    = "/kaggle/temp/.cache"

for p in [os.environ["HF_HOME"], os.environ["HF_DATASETS_CACHE"],
          os.environ["HF_HUB_CACHE"], os.environ["TRANSFORMERS_CACHE"],
          os.environ["XDG_CACHE_HOME"]]:
    os.makedirs(p, exist_ok=True)

!rm -rf /kaggle/working/.cache /kaggle/working/hf
print("Cache dirs ready.")

Cache dirs ready.


In [2]:
!pip -q install -U transformers datasets accelerate evaluate jiwer soundfile packaging

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.4/637.4 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 90.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 107.6 MB/s eta 0:00:00


In [3]:
import numpy as np
import pandas as pd
import torch
import evaluate
import json
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from itertools import groupby

from datasets import load_dataset, Audio
from transformers import (
    WavLMForCTC,
    Wav2Vec2Processor,
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    EarlyStoppingCallback,
)
import transformers
print("transformers:", transformers.__version__)
print("torch:", torch.__version__)
print("GPU:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

transformers: 5.5.0
torch: 2.10.0+cu128
GPU: True
GPU name: Tesla T4


## Step 1 — Load LibriSpeech clean-100

In [4]:
# ── Load LibriSpeech clean-100 ──
# This is ~100 hours of clean read English speech from ~250 speakers
ds = load_dataset(
    "librispeech_asr",
    "clean",
    split="train.100",
    cache_dir=os.environ["HF_DATASETS_CACHE"],
)

print(f"LibriSpeech clean-100: {len(ds)} utterances")
print(f"Features: {ds.features}")

# Check a sample
sample = ds[0]
print(f"\nSample text: {sample['text']}")
print(f"Sample speaker: {sample['speaker_id']}")
print(f"Sample audio SR: {sample['audio']['sampling_rate']}")
print(f"Sample audio length: {len(sample['audio']['array'])} samples")

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

clean/test/0000.parquet:   0%|          | 0.00/350M [00:00<?, ?B/s]

clean/train.100/0000.parquet:   0%|          | 0.00/470M [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/2620 [00:00<?, ? examples/s]

Generating train.100 split:   0%|          | 0/28539 [00:00<?, ? examples/s]

Generating train.360 split:   0%|          | 0/104014 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2703 [00:00<?, ? examples/s]

LibriSpeech clean-100: 28539 utterances
Features: {'file': Value('string'), 'audio': Audio(sampling_rate=16000, decode=True, num_channels=None, stream_index=None), 'text': Value('string'), 'speaker_id': Value('int64'), 'chapter_id': Value('int64'), 'id': Value('string')}

Sample text: CHAPTER SIXTEEN I MIGHT HAVE TOLD YOU OF THE BEGINNING OF THIS LIAISON IN A FEW LINES BUT I WANTED YOU TO SEE EVERY STEP BY WHICH WE CAME I TO AGREE TO WHATEVER MARGUERITE WISHED
Sample speaker: 374
Sample audio SR: 16000
Sample audio length: 232480 samples


In [5]:
# ── Create a small validation split ──
# Take 2% for validation (~570 utterances)
ds_split = ds.train_test_split(test_size=0.02, seed=42)
train_ds = ds_split["train"]
val_ds = ds_split["test"]

print(f"Train: {len(train_ds)} utterances")
print(f"Val:   {len(val_ds)} utterances")

Train: 27968 utterances
Val:   571 utterances


## Step 2 — Build vocabulary and processor

Use the SAME vocab as TORGO (a-z, apostrophe, space, [UNK], [PAD]).
This ensures the B1 CTC head is directly compatible with B2 TORGO fine-tuning.

In [6]:
# ── Build the same vocab as TORGO notebook ──
# We use the same character set so the CTC head is directly reusable
vocab_chars = list("abcdefghijklmnopqrstuvwxyz'")

vocab_dict = {char: idx for idx, char in enumerate(vocab_chars)}
vocab_dict["|"] = len(vocab_dict)   # word delimiter (space)
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

VOCAB_PATH = "/kaggle/working/vocab.json"
with open(VOCAB_PATH, "w") as f:
    json.dump(vocab_dict, f)

print("Vocab:", vocab_dict)
print(f"Vocab size: {len(vocab_dict)}")

Vocab: {'a': 0, 'b': 1, 'c': 2, 'd': 3, 'e': 4, 'f': 5, 'g': 6, 'h': 7, 'i': 8, 'j': 9, 'k': 10, 'l': 11, 'm': 12, 'n': 13, 'o': 14, 'p': 15, 'q': 16, 'r': 17, 's': 18, 't': 19, 'u': 20, 'v': 21, 'w': 22, 'x': 23, 'y': 24, 'z': 25, "'": 26, '|': 27, '[UNK]': 28, '[PAD]': 29}
Vocab size: 30


In [7]:
# ── Build processor ──
tokenizer = Wav2Vec2CTCTokenizer(
    VOCAB_PATH,
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
)

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True,
)

processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
)

PROCESSOR_SAVE_PATH = "/kaggle/working/b1_processor"
processor.save_pretrained(PROCESSOR_SAVE_PATH)
print(f"Processor saved to {PROCESSOR_SAVE_PATH}")
print(f"Vocab size: {len(processor.tokenizer)}")
print(f"Pad token id: {processor.tokenizer.pad_token_id}")

Processor saved to /kaggle/working/b1_processor
Vocab size: 32
Pad token id: 29


## Step 3 — Preprocessing

In [8]:
# ── Filter long audio (>15s) to avoid OOM ──
MAX_AUDIO_SAMPLES = 240_000  # 15 seconds at 16kHz

def filter_long_audio(example):
    return len(example["audio"]["array"]) <= MAX_AUDIO_SAMPLES

before_train = len(train_ds)
before_val = len(val_ds)
train_ds = train_ds.filter(filter_long_audio, num_proc=1)
val_ds = val_ds.filter(filter_long_audio, num_proc=1)

print(f"Train: {before_train} -> {len(train_ds)} (filtered {before_train - len(train_ds)})")
print(f"Val:   {before_val} -> {len(val_ds)} (filtered {before_val - len(val_ds)})")

Filter (num_proc=1):   0%|          | 0/27968 [00:00<?, ? examples/s]

Filter (num_proc=1):   0%|          | 0/571 [00:00<?, ? examples/s]

Train: 27968 -> 19944 (filtered 8024)
Val:   571 -> 392 (filtered 179)


In [9]:
# ── Preprocessing function ──
def prepare_example(batch):
    audio = batch["audio"]

    # Process audio
    inputs = processor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
    )
    batch["input_values"] = inputs.input_values[0]

    # Process text — LibriSpeech text is uppercase, convert to lowercase
    text = batch["text"].lower().strip()
    batch["labels"] = processor.tokenizer(text).input_ids

    return batch

# Apply preprocessing
cols_to_remove = [c for c in train_ds.column_names if c not in []]

train_p = train_ds.map(
    prepare_example,
    remove_columns=train_ds.column_names,
    num_proc=1,
    desc="Preprocessing train",
)

val_p = val_ds.map(
    prepare_example,
    remove_columns=val_ds.column_names,
    num_proc=1,
    desc="Preprocessing val",
)

print(f"Train preprocessed: {len(train_p)}")
print(f"Val preprocessed:   {len(val_p)}")

# Sanity check
sample = train_p[0]
print(f"\nSample input_values length: {len(sample['input_values'])}")
print(f"Sample labels: {sample['labels'][:20]}")
print(f"Sample decoded: {processor.tokenizer.decode(sample['labels'])}")

Preprocessing train (num_proc=1):   0%|          | 0/19944 [00:00<?, ? examples/s]

Preprocessing val (num_proc=1):   0%|          | 0/392 [00:00<?, ? examples/s]

Train preprocessed: 19944
Val preprocessed:   392

Sample input_values length: 168240
Sample labels: [12, 24, 27, 3, 4, 0, 17, 27, 12, 0, 3, 0, 12, 27, 7, 4, 27, 17, 4, 15]
Sample decoded: my dear madam he replied this invitation is particularly gratifying because it is what i have ben hoping to receive


## Step 4 — Data Collator

In [10]:
@dataclass
class DataCollatorCTCWithPadding:
    processor: Any
    padding: Union[bool, str] = "longest"

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [
            {"input_values": f["input_values"]}
            for f in features
        ]
        batch = self.processor.feature_extractor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features,
            padding=self.padding,
            return_tensors="pt",
        )
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor)
print("Data collator ready.")

Data collator ready.


## Step 5 — Load WavLM and freeze everything except CTC head

In [11]:
MODEL_NAME = "microsoft/wavlm-base"

model = WavLMForCTC.from_pretrained(
    MODEL_NAME,
    cache_dir=os.environ["TRANSFORMERS_CACHE"],
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
    ctc_zero_infinity=True,
)

# ── FREEZE EVERYTHING ──
# Freeze CNN feature encoder
model.freeze_feature_encoder()

# Freeze all transformer encoder layers
for param in model.wavlm.parameters():
    param.requires_grad = False

# Only lm_head (CTC projection layer) remains trainable
for param in model.lm_head.parameters():
    param.requires_grad = True

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}  (CTC head only)")
print(f"Frozen parameters:    {frozen_params:,}  (entire WavLM encoder)")
print(f"\nTrainable param names:")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"  {name}: {param.shape}")

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

WavLMForCTC LOAD REPORT from: microsoft/wavlm-base
Key            | Status  | 
---------------+---------+-
lm_head.weight | MISSING | 
lm_head.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total parameters:     94,406,544
Trainable parameters: 24,608  (CTC head only)
Frozen parameters:    94,381,936  (entire WavLM encoder)

Trainable param names:
  lm_head.weight: torch.Size([32, 768])
  lm_head.bias: torch.Size([32])


## Step 6 — Metrics

In [12]:
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def decode_ctc_predictions(pred_ids, tokenizer):
    blank_id = tokenizer.pad_token_id
    decoded = []
    for seq in pred_ids:
        collapsed = [k for k, _ in groupby(seq)]
        filtered = [t for t in collapsed if t != blank_id]
        if len(filtered) == 0:
            decoded.append("")
        else:
            decoded.append(tokenizer.decode(filtered, skip_special_tokens=True))
    return decoded

def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)
    label_ids = pred.label_ids

    label_ids = np.where(
        label_ids != -100,
        label_ids,
        processor.tokenizer.pad_token_id
    )

    pred_str = decode_ctc_predictions(pred_ids, processor.tokenizer)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    pred_str = [p.strip() for p in pred_str]
    label_str = [l.strip() for l in label_str]

    eval_preds = [p if len(p) > 0 else "\u27e8empty\u27e9" for p in pred_str]
    eval_labels = [l if len(l) > 0 else "\u27e8empty\u27e9" for l in label_str]

    wer = wer_metric.compute(predictions=eval_preds, references=eval_labels)
    cer = cer_metric.compute(predictions=eval_preds, references=eval_labels)

    return {"wer": round(wer, 4), "cer": round(cer, 4)}

print("Metrics ready.")

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Metrics ready.


## Step 7 — Training

Since only the CTC head (~22K params) is trainable:
- Higher learning rate (1e-3) — this is a randomly initialized linear layer
- Larger batch size — very little GPU memory used for gradients
- Fewer epochs needed — simple linear layer converges fast on 100h of data

In [13]:
CHECKPOINT_DIR = "/kaggle/working/b1_ctc_head"

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,

    # Training schedule
    num_train_epochs=5,                  # head converges fast on 100h
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,       # effective batch = 16
    learning_rate=1e-3,                  # high LR for randomly initialized head
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    weight_decay=0.0,                    # no regularization for tiny layer

    # Eval + checkpointing
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,

    # Memory
    fp16=True,
    dataloader_num_workers=2,

    # Logging
    logging_steps=50,
    logging_dir="/kaggle/temp/tb_logs",
    report_to="none",

    # Save space
    save_total_limit=2,
)

print("Training arguments set.")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training arguments set.
Effective batch size: 16


In [14]:
# ── Callbacks ──
class PrintLossCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        if "loss" in logs:
            print(f"[step {state.global_step}] train_loss={logs['loss']:.4f}")
        if "learning_rate" in logs:
            print(f"[step {state.global_step}] lr={logs['learning_rate']:.2e}")
        if "eval_cer" in logs:
            print(f"[step {state.global_step}] val_cer={logs['eval_cer']:.4f}  val_wer={logs.get('eval_wer', 'N/A')}")

class SanityCheckCallback(TrainerCallback):
    def on_evaluate(self, args, state, control, model=None, **kwargs):
        if model is None:
            return
        model.eval()
        device = next(model.parameters()).device
        try:
            sample = val_p[0]
            input_values = torch.tensor(sample["input_values"]).unsqueeze(0).to(device)
            with torch.no_grad():
                logits = model(input_values=input_values).logits
            pred_ids = torch.argmax(logits, dim=-1).cpu().numpy()
            pred_text = decode_ctc_predictions(pred_ids, processor.tokenizer)[0]
            label_ids = [x for x in sample["labels"] if x != -100]
            ref_text = processor.tokenizer.decode(label_ids, skip_special_tokens=True)
            print(f"  [SANITY CHECK] REF : {ref_text}")
            print(f"  [SANITY CHECK] PRED: {pred_text}")
        except Exception as e:
            print(f"  [SANITY CHECK] Error: {e}")

early_stopping = EarlyStoppingCallback(
    early_stopping_patience=3,
    early_stopping_threshold=0.001,
)

print("Callbacks ready.")

Callbacks ready.


In [15]:
# ── Train ──
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_p,
    eval_dataset=val_p,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[PrintLossCallback(), SanityCheckCallback(), early_stopping],
)

print("\n--- DISK USAGE BEFORE TRAINING ---")
!du -sh /kaggle/working/* 2>/dev/null || true

print("\nStarting B1 training (CTC head only on LibriSpeech)...")
trainer.train()


--- DISK USAGE BEFORE TRAINING ---
4.0K	/kaggle/working/b1_ctc_head
20K	/kaggle/working/b1_processor
76K	/kaggle/working/__notebook__.ipynb
4.0K	/kaggle/working/vocab.json

Starting B1 training (CTC head only on LibriSpeech)...


/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


Epoch,Training Loss,Validation Loss,Wer,Cer
1,1.762268,0.758490,0.666400,0.215800
2,1.652327,0.670502,0.614900,0.187200
3,1.516317,0.637226,0.599000,0.178900
4,1.459916,0.621683,0.584400,0.173700
5,1.485667,0.618392,0.583100,0.172500


[step 50] train_loss=18.1065
[step 50] lr=7.85e-05
[step 100] train_loss=16.6333
[step 100] lr=1.59e-04
[step 150] train_loss=13.8916
[step 150] lr=2.39e-04
[step 200] train_loss=10.8551
[step 200] lr=3.19e-04
[step 250] train_loss=7.2899
[step 250] lr=3.99e-04
[step 300] train_loss=5.4899
[step 300] lr=4.79e-04
[step 350] train_loss=4.1264
[step 350] lr=5.59e-04
[step 400] train_loss=3.1484
[step 400] lr=6.39e-04
[step 450] train_loss=3.1286
[step 450] lr=7.20e-04
[step 500] train_loss=2.5962
[step 500] lr=8.00e-04
[step 550] train_loss=2.3686
[step 550] lr=8.80e-04
[step 600] train_loss=2.2103
[step 600] lr=9.60e-04
[step 650] train_loss=2.2689
[step 650] lr=9.96e-04
[step 700] train_loss=2.1035
[step 700] lr=9.87e-04
[step 750] train_loss=2.0882
[step 750] lr=9.78e-04
[step 800] train_loss=2.1255
[step 800] lr=9.69e-04
[step 850] train_loss=1.9710
[step 850] lr=9.60e-04
[step 900] train_loss=1.8591
[step 900] lr=9.51e-04
[step 950] train_loss=1.8658
[step 950] lr=9.42e-04
[step 1000

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


[step 5000] train_loss=1.4753
[step 5000] lr=2.20e-04
[step 5050] train_loss=1.5202
[step 5050] lr=2.11e-04
[step 5100] train_loss=1.4367
[step 5100] lr=2.02e-04
[step 5150] train_loss=1.5392
[step 5150] lr=1.94e-04
[step 5200] train_loss=1.4830
[step 5200] lr=1.85e-04
[step 5250] train_loss=1.5184
[step 5250] lr=1.76e-04
[step 5300] train_loss=1.4796
[step 5300] lr=1.67e-04
[step 5350] train_loss=1.4868
[step 5350] lr=1.58e-04
[step 5400] train_loss=1.4923
[step 5400] lr=1.49e-04
[step 5450] train_loss=1.4658
[step 5450] lr=1.40e-04
[step 5500] train_loss=1.4688
[step 5500] lr=1.31e-04
[step 5550] train_loss=1.4735
[step 5550] lr=1.22e-04
[step 5600] train_loss=1.4672
[step 5600] lr=1.13e-04
[step 5650] train_loss=1.5319
[step 5650] lr=1.04e-04
[step 5700] train_loss=1.4951
[step 5700] lr=9.55e-05
[step 5750] train_loss=1.5194
[step 5750] lr=8.66e-05
[step 5800] train_loss=1.4606
[step 5800] lr=7.77e-05
[step 5850] train_loss=1.4358
[step 5850] lr=6.88e-05
[step 5900] train_loss=1.492

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6235, training_loss=2.1554796722093963, metrics={'train_runtime': 6754.3105, 'train_samples_per_second': 14.764, 'train_steps_per_second': 0.923, 'total_flos': 1.3237955181608516e+19, 'train_loss': 2.1554796722093963, 'epoch': 5.0})

## Step 8 — Save B1 model

Save the full model (frozen WavLM + trained CTC head).
B2 notebook will load this and unfreeze the encoder for TORGO fine-tuning.

In [16]:
# ── Save ──
FINAL_MODEL_PATH = "/kaggle/working/b1_wavlm_ctc_librispeech"
trainer.save_model(FINAL_MODEL_PATH)
processor.save_pretrained(FINAL_MODEL_PATH)

print(f"B1 model saved to {FINAL_MODEL_PATH}")

print("\n--- DISK USAGE ---")
!du -sh /kaggle/working/* 2>/dev/null || true

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

B1 model saved to /kaggle/working/b1_wavlm_ctc_librispeech

--- DISK USAGE ---
721M	/kaggle/working/b1_ctc_head
20K	/kaggle/working/b1_processor
361M	/kaggle/working/b1_wavlm_ctc_librispeech
104K	/kaggle/working/__notebook__.ipynb
4.0K	/kaggle/working/vocab.json


In [17]:
# ── Zip for download ──
!zip -r /kaggle/working/b1_wavlm_ctc_librispeech.zip /kaggle/working/b1_wavlm_ctc_librispeech/
print("Zip ready for download.")

  adding: kaggle/working/b1_wavlm_ctc_librispeech/ (stored 0%)
  adding: kaggle/working/b1_wavlm_ctc_librispeech/vocab.json (deflated 56%)
  adding: kaggle/working/b1_wavlm_ctc_librispeech/training_args.bin (deflated 53%)
  adding: kaggle/working/b1_wavlm_ctc_librispeech/added_tokens.json (deflated 20%)
  adding: kaggle/working/b1_wavlm_ctc_librispeech/config.json (deflated 66%)
  adding: kaggle/working/b1_wavlm_ctc_librispeech/model.safetensors (deflated 41%)
  adding: kaggle/working/b1_wavlm_ctc_librispeech/tokenizer_config.json (deflated 73%)
  adding: kaggle/working/b1_wavlm_ctc_librispeech/processor_config.json (deflated 44%)
Zip ready for download.


## Step 9 — Quick evaluation on LibriSpeech val

In [18]:
# ── Quick eval on validation set ──
model.eval()
device = next(model.parameters()).device

sample_preds = []
sample_labels = []

for idx in range(min(50, len(val_p))):
    sample = val_p[idx]
    input_values = torch.tensor(sample["input_values"]).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(input_values=input_values).logits

    pred_ids = torch.argmax(logits, dim=-1).cpu().numpy()
    pred_text = decode_ctc_predictions(pred_ids, processor.tokenizer)[0]

    label_ids = [x for x in sample["labels"] if x != -100]
    ref_text = processor.tokenizer.decode(label_ids, skip_special_tokens=True)

    sample_preds.append(pred_text.strip())
    sample_labels.append(ref_text.strip())

eval_preds = [p if len(p) > 0 else "\u27e8empty\u27e9" for p in sample_preds]
wer = wer_metric.compute(predictions=eval_preds, references=sample_labels)
cer = cer_metric.compute(predictions=eval_preds, references=sample_labels)

print(f"B1 LibriSpeech validation (50 samples):")
print(f"  WER: {wer*100:.2f}%")
print(f"  CER: {cer*100:.2f}%")

print(f"\nSample predictions:")
print("-" * 70)
for i in range(min(15, len(sample_preds))):
    print(f"REF : {sample_labels[i]}")
    print(f"PRED: {sample_preds[i]}")
    print()

B1 LibriSpeech validation (50 samples):
  WER: 58.70%
  CER: 17.10%

Sample predictions:
----------------------------------------------------------------------
REF : before she started to kep her apointment with the solicitors
PRED: be for she stardted to keper a poiment wit the sulisiters

REF : and fancied his countenance was not altogether unknown to me i asked him some questions concerning his family and his country but al the answers i could get were sighs and tears i tok pity on him
PRED: and fansed his countinans was notal to gether unown tomi askimsomd qestons con serning his famly in his counrybut al the ancers i oue get wer sis an tersi tok pit on him

REF : the weirdnes in which mily's absence had left them deficient she made it inded efective for him by sudenly adresing him you know nothing sir but not the least litle bit about my friend
PRED: thewardnesamn which mily's apsns had lef them dfitionshe mad eid in ded fectoif for him by csudly a dresing him you no nothing srbut

In [19]:
print("\n" + "=" * 55)
print("B1 COMPLETE")
print("=" * 55)
print(f"Model: WavLM-base (frozen) + CTC head (trained)")
print(f"Trained on: LibriSpeech clean-100 (~28K utterances)")
print(f"Saved to: {FINAL_MODEL_PATH}")
print(f"\nNext step: Load B1 in B2 notebook, unfreeze encoder,")
print(f"fine-tune on TORGO for dysarthric speech adaptation.")
print(f"\nIn B2, change MODEL_NAME to:")
print(f'  MODEL_PATH = "/kaggle/input/b1-wavlm-ctc-librispeech/b1_wavlm_ctc_librispeech"')


B1 COMPLETE
Model: WavLM-base (frozen) + CTC head (trained)
Trained on: LibriSpeech clean-100 (~28K utterances)
Saved to: /kaggle/working/b1_wavlm_ctc_librispeech

Next step: Load B1 in B2 notebook, unfreeze encoder,
fine-tune on TORGO for dysarthric speech adaptation.

In B2, change MODEL_NAME to:
  MODEL_PATH = "/kaggle/input/b1-wavlm-ctc-librispeech/b1_wavlm_ctc_librispeech"
